# Laboratorium 5

Natalia Ślepowrońska 318847

W tym ćwiczeniu przeanalizowano działanie różnych metod detekcji punktów charakterystycznych w zależności od sceny oraz działanie metod dla określonej sceny w zależności od wartości parametrów tych metod.

### Przygotowanie środowiska

Pierwszy krok: Instalacja zależności.

In [2]:
!pip install opencv-python opencv-contrib-python matplotlib torch torchvision
!git clone https://github.com/magicleap/SuperGluePretrainedNetwork.git
%cd SuperGluePretrainedNetwork
!sed -i 's/opencv-python==4.1.2.30/opencv-python/' requirements.txt
!pip install -r requirements.txt

Cloning into 'SuperGluePretrainedNetwork'...
remote: Enumerating objects: 185, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 185 (delta 0), reused 0 (delta 0), pack-reused 184 (from 2)
Receiving objects: 100% (185/185), 118.85 MiB | 29.60 MiB/s, done.
Resolving deltas: 100% (52/52), done.
Updating files: 100% (89/89), done.
/content/SuperGluePretrainedNetwork/SuperGluePretrainedNetwork


Drugi krok: Import bibliotek.

In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import os
import torch
import torch
from pathlib import Path
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from models.matching import Matching
from models.utils import frame2tensor, read_image, make_matching_plot_fast

Trzeci krok: Rozpakowanie datasetu.

In [ ]:
%cd /content
!unzip hpatches-sequences-release

In [ ]:
katalog_hpatches = Path("/content/hpatches-sequences-release")

### Metody wyznaczania punktów charakterystycznych: SIFT, SIFT+FLANN, ORB, AKAZE, SuperGlue

Poniższa funkcja realizuje detekcję i dopasowanie punktów charakterystycznych metodą SIFT, wykorzystując algorytm uczenia maszynowego k‑NN (k=2 sąsiadów) oraz filtrację dopasowań testem Lowe’a (z zadanym ratio), z opcjonalną wizualizacją wyników.

In [ ]:
def run_sift(img1_gray, img2_gray, ratio=0.75, visualize=True):
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(img1_gray, None)
    kp2, des2 = sift.detectAndCompute(img2_gray, None)

    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des1, des2, k=2)

    good_matches = []
    for m, n in matches:
        if m.distance < ratio * n.distance:
            good_matches.append(m)

    len_good_matches = len(good_matches)
    print(f"SIFT: {len_good_matches} matches")

    if visualize:
        img = cv2.drawMatches(img1_gray, kp1, img2_gray, kp2,
                              good_matches, None,
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(15, 8))
        plt.imshow(img)
        plt.title(f"SIFT ({len_good_matches} matches)")
        plt.axis('off')
        plt.show()

    return kp1, kp2, good_matches

Ponisza funkcja realizuje detekcję punktów metodą SIFT, dopasowanie z użyciem przybliżonego wyszukiwania (FLANN, k‑NN k=2) oraz filtrację dopasowań testem Lowe’a (ratio), z opcjonalną wizualizacją wyników.

In [ ]:
def run_sift_flann(img1_gray, img2_gray, ratio=0.75, visualize=True):
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(img1_gray, None)
    kp2, des2 = sift.detectAndCompute(img2_gray, None)

    FLANN_INDEX_KDTREE = 1
    index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)

    flann = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.knnMatch(des1, des2, k=2)

    good_matches = []
    for m, n in matches:
        if m.distance < ratio * n.distance:
            good_matches.append(m)

    len_good_matches = len(good_matches)
    print(f"SIFT+FLANN: {len_good_matches} matches")

    if visualize:
        img = cv2.drawMatches(img1_gray, kp1, img2_gray, kp2,
                              good_matches[:50], None,
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(15, 8))
        plt.imshow(img)
        plt.title(f"SIFT + FLANN ({len_good_matches} matches)")
        plt.axis('off')
        plt.show()

    return kp1, kp2, good_matches

Poniższa funkcja realizuje detekcję i opis punktów metodą ORB, dopasowanie przy użyciu brute‑force (metryka Hamming, z crossCheck), sortowanie dopasowań według odległości oraz opcjonalną wizualizację wyników.

In [ ]:
def run_orb(img1_gray, img2_gray, nfeatures=500, fasttreshold=20, visualize=True):
    orb = cv2.ORB_create(nfeatures=nfeatures, fastThreshold=fasttreshold)
    kp1, des1 = orb.detectAndCompute(img1_gray, None)
    kp2, des2 = orb.detectAndCompute(img2_gray, None)

    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)

    matches = sorted(matches, key=lambda x: x.distance)
    len_good_matches = len(matches)
    print(f"ORB: {len_good_matches} matches")

    if visualize:
        img = cv2.drawMatches(img1_gray, kp1, img2_gray, kp2,
                              matches, None,
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(15, 8))
        plt.imshow(img)
        plt.title(f"ORB ({len_good_matches} matches)")
        plt.axis('off')
        plt.show()

    return kp1, kp2, matches

Poniższa funkcja realizuje detekcję i opis punktów metodą AKAZE, dopasowanie przy użyciu brute‑force (metryka Hamming z crossCheck), sortowanie dopasowań według odległości oraz opcjonalną wizualizację wyników.

In [ ]:
def run_akaze(img1_gray, img2_gray, noctaves=4, treshold=0.001, visualize=True):
    akaze = cv2.AKAZE_create(nOctaves=noctaves, threshold=treshold)
    kp1, des1 = akaze.detectAndCompute(img1_gray, None)
    kp2, des2 = akaze.detectAndCompute(img2_gray, None)

    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)

    matches = sorted(matches, key=lambda x: x.distance)
    len_good_matches = len(matches)
    print(f"AKAZE: {len_good_matches} matches")

    if visualize:
        img = cv2.drawMatches(img1_gray, kp1, img2_gray, kp2,
                              matches, None,
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(15, 8))
        plt.imshow(img)
        plt.title(f"AKAZE ({len_good_matches} matches)")
        plt.axis('off')
        plt.show()

    return kp1, kp2, matches


Poniższa funkcja realizuje detekcję punktów (SuperPoint) i ich dopasowanie metodą SuperGlue z użyciem sieci neuronowej, konwertuje wyniki do formatu OpenCV oraz opcjonalnie wizualizuje dopasowania.

In [ ]:
def run_superglue(img1_path, img2_path, matching, device, visualize=True):
    image0, _, _ = read_image(str(img1_path), device, [640, 480], 0, False)
    image1, _, _ = read_image(str(img2_path), device, [640, 480], 0, False)

    pred = matching({
        'image0': frame2tensor(image0, device),
        'image1': frame2tensor(image1, device)
    })

    kpts0 = pred['keypoints0'][0].cpu().numpy()
    kpts1 = pred['keypoints1'][0].cpu().numpy()
    matches0 = pred['matches0'][0].cpu().numpy()

    valid = matches0 > -1

    # konwersja do formatu CV2
    kp1 = [cv2.KeyPoint(x=float(p[0]), y=float(p[1]), size=1) for p in kpts0]
    kp2 = [cv2.KeyPoint(x=float(p[0]), y=float(p[1]), size=1) for p in kpts1]

    matches = []
    for i, m in enumerate(matches0):
        if m > -1:
            matches.append(cv2.DMatch(_queryIdx=i,
                                     _trainIdx=int(m),
                                     _distance=0))

    len_good_matches = len(matches)
    print(f"SuperGlue: {len_good_matches} matches")

    if visualize:
        img1 = cv2.imread(str(img1_path), cv2.IMREAD_GRAYSCALE)
        img2 = cv2.imread(str(img2_path), cv2.IMREAD_GRAYSCALE)

        img = cv2.drawMatches(img1, kp1, img2, kp2,
                              matches, None,
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(15, 8))
        plt.imshow(img)
        plt.title(f"SuperGlue ({len_good_matches} matches)")
        plt.axis('off')
        plt.show()

    return kp1, kp2, matches

### Weryfikacja dopasowań: Precision, Recall, AUC, MMA

Poniższa funkcja ocenia jakość dopasowań poprzez obliczenie błędu reprojekcji z użyciem homografii, a następnie wyznacza Recall i MMA na podstawie liczby dopasowań spełniających zadany próg błędu.

In [ ]:
def evaluate_matches(kp1, kp2, matches, homography, threshold=3):
    true_pos = 0
    total = len(matches)
    all_preds, all_labels = [], []
    for m in matches:
        pt1 = np.array(kp1[m.queryIdx].pt + (1,)).reshape(3,1)
        pt2 = np.array(kp2[m.trainIdx].pt).reshape(2)
        proj = homography @ pt1
        proj = (proj[:2] / proj[2]).flatten()
        error = np.linalg.norm(proj - pt2)
        all_preds.append(1 if error < threshold else 0)
        all_labels.append(1)
        if error < threshold:
            true_pos += 1
    #precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    #auc = roc_auc_score(all_labels, all_preds)
    mma = true_pos / total if total > 0 else 0
    return recall, mma

Recall to miara określająca jaki procent dopasowań został uznany za poprawny.
MMA to miara określająca dokładność dopasowań na podstawie błędu reprojekcji.

Obie miary sprawdzają, czy błędy dopasowań są poniżej zadanego progu. Pozwolą one na ocenę działania metod detekcji punktów charakterystycznych.

### Porównanie metod

Poniższa funkcja przeprowadza porównanie jakości dopasowań punktów charakterystycznych dla różnych metod, wykorzystując metryki Recall i MMA.

In [ ]:
def compare(img1_path, img2_path, H, matching, device, sift_ratio=0.75, flann_ratio=0.75, nfeatures=500, noctaves=4, fasttreshold=20, treshold=0.001, error_thresh=3, visualize=True):

    img1_gray = cv2.imread(str(img1_path), cv2.IMREAD_GRAYSCALE);
    img2_gray = cv2.imread(str(img2_path), cv2.IMREAD_GRAYSCALE)

    kp1_sift, kp2_sift, matches_sift = run_sift(img1_gray, img2_gray, ratio=sift_ratio, visualize=visualize)
    r_sift, mma_sift = evaluate_matches(kp1_sift, kp2_sift, matches_sift, H, threshold=error_thresh)

    kp1_flann, kp2_flann, matches_flann = run_sift_flann(img1_gray, img2_gray, ratio=flann_ratio, visualize=visualize)
    r_flann, mma_flann = evaluate_matches(kp1_flann, kp2_flann, matches_flann, H, threshold=error_thresh)

    kp1_orb, kp2_orb, matches_orb = run_orb(img1_gray, img2_gray, nfeatures=nfeatures, fasttreshold=fasttreshold, visualize=visualize)
    r_orb, mma_orb = evaluate_matches(kp1_orb, kp2_orb, matches_orb, H, threshold=error_thresh)

    kp1_akaze, kp2_akaze, matches_akaze = run_akaze(img1_gray, img2_gray, noctaves=noctaves, treshold=treshold,visualize=visualize)
    r_akaze, mma_akaze = evaluate_matches(kp1_akaze, kp2_akaze, matches_akaze, H, threshold=error_thresh)

    kp1_sg, kp2_sg, matches_sg = run_superglue(img1_path, img2_path, matching, device, visualize=visualize)
    r_sg, mma_sg = evaluate_matches(kp1_sg, kp2_sg, matches_sg, H, threshold=error_thresh)

    print(f"SIFT       -> Recall: {r_sift:.2f}, MMA: {mma_sift:.2f}")
    print(f"SIFT+FLANN -> Recall: {r_flann:.2f}, MMA: {mma_flann:.2f}")
    print(f"ORB        -> Recall: {r_orb:.2f}, MMA: {mma_orb:.2f}")
    print(f"AKAZE      -> Recall: {r_akaze:.2f}, MMA: {mma_akaze:.2f}")
    print(f"SuperGlue  -> Recall: {r_sg:.2f}, MMA: {mma_sg:.2f}")



Wyłączono niektóre warningi dla większej przejrzystości wyników.

In [ ]:
import warnings
from sklearn.exceptions import UndefinedMetricWarning

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
matching = Matching({
    'superpoint': {"nms_radius": 4, "keypoint_threshold": 0.005},
    'superglue': {"weights": 'outdoor'}
}).eval().to(device)

Loaded SuperPoint model
Loaded SuperGlue model ("outdoor" weights)


Analizy przeprowadzono najpierw z wizualizacjami, następnie bez i taki notatnik zapisano. Ze względu na zmianę lokalizacji i problemy z połączeniem internetowym, nie udało się ponownie wgrać zip do środowiska, aby wykonać wizualizacje w ostatecznej wersji notatnika.Przetestowano jednak, że funkcjonalność ta działa i przeanalizowano wcześniej wizualizacje działania metod. We wnioskach uwzględniono wizualne zmiany między klatkami. Laboratorium wykonano w google colab ze względu na problemy z instalacją wszystkich niezbędnych zależności lokalnie na urządzeniu, jednak w google colab jakość połączenia internetowego okazała się bardzo ważna.

#### Łódź

Pierwszą analizę przeprowadzono dla zmiany perspektywy widoku łodzi (czarno biała scena, drobne szczegóły) - względem klatki 1, na klatce 2 widać zmianę orientacji kamery.

In [ ]:
img1_path = katalog_hpatches / "v_boat" / "1.ppm"
img2_path = katalog_hpatches / "v_boat" / "2.ppm"

H = np.loadtxt(img1_path.parent / 'H_1_2')

results1 = compare(
    img1_path,
    img2_path,
    H,
    matching,
    device,
    visualize=False
)

SIFT: 2341 matches
SIFT+FLANN: 2352 matches
ORB: 261 matches
AKAZE: 2485 matches
SuperGlue: 703 matches
SIFT       -> Recall: 0.97, MMA: 0.97
SIFT+FLANN -> Recall: 0.96, MMA: 0.96
ORB        -> Recall: 0.89, MMA: 0.89
AKAZE      -> Recall: 0.84, MMA: 0.84
SuperGlue  -> Recall: 0.00, MMA: 0.00


SIFT i SIFT+FLANN osiągają najwyższe wartości Recall przy dużej liczbie dopasowań, co wskazuje na wysoką skuteczność oraz stabilność tych metod w warunkach zmiany perspektywy i skali. AKAZE generuje największą liczbę dopasowań, jednak przy niższej wartości Recall (0.84), co sugeruje obecność większej liczby błędnych dopasowań. ORB wykrywa znacznie mniej dopasowań, ale zachowuje stosunkowo wysoką jakość (0.89), co wskazuje na jego selektywny charakter i kompromis między dokładnością a liczbą punktów.

W przypadku SuperGlue uzyskano Recall równy 0, mimo umiarkowanej liczby dopasowań, co wskazuje na problem implementacyjny, najprawdopodobniej wynikający z niespójności skali obrazów lub niepoprawnego wykorzystania homografii.

Kolejną analizę przeprowadzono dla zmiany perspektywy widoku łodzi (czarno biała scena, drobne szczegóły) - względem klatki 1, na klatce 5 widać zmianę odległości od obiektu i orientacji kamery.

In [ ]:
img1_path = katalog_hpatches / "v_boat" / "1.ppm"
img2_path = katalog_hpatches / "v_boat" / "5.ppm"

H = np.loadtxt(img1_path.parent / 'H_1_5')

results1 = compare(
    img1_path,
    img2_path,
    H,
    matching,
    device,
    visualize=False
)

SIFT: 704 matches
SIFT+FLANN: 725 matches
ORB: 185 matches
AKAZE: 1204 matches
SuperGlue: 23 matches
SIFT       -> Recall: 0.86, MMA: 0.86
SIFT+FLANN -> Recall: 0.83, MMA: 0.83
ORB        -> Recall: 0.66, MMA: 0.66
AKAZE      -> Recall: 0.51, MMA: 0.51
SuperGlue  -> Recall: 0.00, MMA: 0.00


Przy większym odstępie między klatkami i większymi różnicami między ujęciem łodzi, spadła liczba dopasowań i jakość dla wszystkich metod, co potwierdza większą złożoność problemu (większa zmiana perspektywy).

SIFT i SIFT+FLANN nadal osiągają najlepsze wyniki (Recall 0.83–0.86) - są najbardziej odporne na zmiany sceny.
ORB wyraźnie traci (Recall 0.66), co pokazuje jego ograniczoną odporność na większe transformacje.
AKAZE generuje majwięcej dopasowań, ale o niższej jakości (Recall 0.51).
SuperGlue zwraca bardzo mało dopasowań (23) i nadal Recall = 0.

#### Most

Kolejną analizę przeprowadzono dla sceny mostu nad strumieniem (w kolorze), oświetlonego wchodzącym / zachodzącym słońcem (gdzie zperspektywy oglądającego, słońce znajduje się za mostem) - między klatką 1 i 2 widać zwiększenie się intensywności padającego światła słonecznego.

In [ ]:
img1_path = katalog_hpatches / "i_fog" / "1.ppm"
img2_path = katalog_hpatches / "i_fog" / "2.ppm"

H = np.loadtxt(img1_path.parent / 'H_1_2')

results2 = compare(
    img1_path,
    img2_path,
    H,
    matching,
    device,
    visualize=False
)

SIFT: 318 matches
SIFT+FLANN: 322 matches
ORB: 136 matches
AKAZE: 116 matches
SuperGlue: 270 matches
SIFT       -> Recall: 0.99, MMA: 0.99
SIFT+FLANN -> Recall: 0.99, MMA: 0.99
ORB        -> Recall: 0.23, MMA: 0.23
AKAZE      -> Recall: 0.72, MMA: 0.72
SuperGlue  -> Recall: 0.96, MMA: 0.96


SIFT i SIFT+FLANN osiągają bardzo wysoką skuteczność (Recall 0.99) - są praktycznie odporne na zmiany oświetlenia. Metody te bazują na gradientach, dzięki czemu są relatywnie niewrażliwe na zmiany jasności sceny.

SuperGlue również radzi sobie bardzo dobrze (Recall 0.96), potwierdza to wysoką jakość dopasowań oraz odporność na zmiany warunków oświetleniowych.

AKAZE osiąga umiarkowane wyniki (Recall 0.72) - częściowa odporność na zmiany oświetlenia.

ORB wypada bardzo słabo (Recall 0.23), cechuje się silną wrażliwością na zmiany jasności, co wynika z prostych deskryptorów binarnych.

Kolejną analizę wykonano dla klatek 1 i 3 sceny mostu, gdzie na klatce 3 słońce całkowicie przestało padać na most - mogło zostać przesłonięte przez chmury.

In [ ]:
img1_path = katalog_hpatches / "i_fog" / "1.ppm"
img2_path = katalog_hpatches / "i_fog" / "3.ppm"

H = np.loadtxt(img1_path.parent / 'H_1_3')

results2 = compare(
    img1_path,
    img2_path,
    H,
    matching,
    device,
    visualize=False
)

SIFT: 202 matches
SIFT+FLANN: 204 matches
ORB: 127 matches
AKAZE: 78 matches
SuperGlue: 215 matches
SIFT       -> Recall: 0.99, MMA: 0.99
SIFT+FLANN -> Recall: 0.99, MMA: 0.99
ORB        -> Recall: 0.05, MMA: 0.05
AKAZE      -> Recall: 0.55, MMA: 0.55
SuperGlue  -> Recall: 0.96, MMA: 0.96


Większy odstęp między klatkami oraz istotna zmiana oświetlenia (zanik bezpośredniego światła słonecznego) nie wpływają znacząco na działanie metod SIFT i SIFT+FLANN, które utrzymują bardzo wysoką skuteczność (Recall 0.99). Podobnie SuperGlue zachowuje wysoką jakość dopasowań (Recall 0.96), co wskazuje na dużą odporność na zmiany warunków oświetleniowych.

Natomiast wskaźniki dla metod ORB i AKAZE ulegają wyraźnej degradacji. ORB osiąga bardzo niski Recall (0.05), co wskazuje na dużą wrażliwość na zmiany oświetlenia oraz większy odstęp między klatkami. AKAZE również traci na jakości (Recall 0.55), choć jego degradacja jest mniejsza niż w przypadku ORB.

Kolejną analizę wykonano dla klatek 1 i 6 sceny mostu, gdzie na klatce 6 słońce świeci wysoko, a oświetlenie jest rozproszone przez mgłę.

In [ ]:
img1_path = katalog_hpatches / "i_fog" / "1.ppm"
img2_path = katalog_hpatches / "i_fog" / "6.ppm"

H = np.loadtxt(img1_path.parent / 'H_1_6')

results2 = compare(
    img1_path,
    img2_path,
    H,
    matching,
    device,
    visualize=False
)

SIFT: 112 matches
SIFT+FLANN: 113 matches
ORB: 118 matches
AKAZE: 18 matches
SuperGlue: 131 matches
SIFT       -> Recall: 0.96, MMA: 0.96
SIFT+FLANN -> Recall: 0.96, MMA: 0.96
ORB        -> Recall: 0.03, MMA: 0.03
AKAZE      -> Recall: 0.56, MMA: 0.56
SuperGlue  -> Recall: 0.88, MMA: 0.88


Zmiana oświetlenia połączona z dużym odstępem między klatkami powoduje degradację wszystkich metod, przy czym SIFT pozostaje najbardziej stabilny, SuperGlue wykazuje spadek, ale nadal działa dobrze, natomiast ORB i AKAZE tracą istotnie na skuteczności (szczególne ORB).

#### Zamek

Kolejną analizę wykonano dla sceny zamku oświetlonego bezpośrednio przez słońce - oglądający widzi oświetlony front zamku, słońce znajdowałoby się za oglądającym scenę. W porównaniu do klatki 1, na klatce 2 znajduje się mniej cieni - słońce musiało przemieścić się wyżej na niebie, zmienił się kąt padania światła.

In [ ]:
img1_path = katalog_hpatches / "i_fenis" / "1.ppm"
img2_path = katalog_hpatches / "i_fenis" / "2.ppm"

H = np.loadtxt(img1_path.parent / 'H_1_2')

results3 = compare(
    img1_path,
    img2_path,
    H,
    matching,
    device,
    visualize=False
)

SIFT: 406 matches
SIFT+FLANN: 409 matches
ORB: 254 matches
AKAZE: 436 matches
SuperGlue: 520 matches
SIFT       -> Recall: 0.95, MMA: 0.95
SIFT+FLANN -> Recall: 0.95, MMA: 0.95
ORB        -> Recall: 0.85, MMA: 0.85
AKAZE      -> Recall: 0.86, MMA: 0.86
SuperGlue  -> Recall: 0.97, MMA: 0.97


W tym przypadku ponownie metody ORB i AKAZE wypadają gorzej od pozostałych, jednak nie w takim stopniu jak dla poprzedniej sceny. Wynika to z faktu, że zmiana oświetlenia jest mniej drastyczna i nie powoduje tak dużych zmian w strukturze obrazu (np. zaniku kontrastu czy dużych różnic jasności).

SIFT i SIFT+FLANN utrzymują wysoką skuteczność dopasowań (Recall 0.95), co potwierdza ich odporność na zmiany kąta padania światła i umiarkowane zmiany cieni.

SuperGlue osiąga najwyższą jakość dopasowań (Recall 0.97), co wynika z wykorzystania kontekstu globalnego i lepszego modelowania relacji między punktami.

ORB i AKAZE osiągają zbliżone wyniki (Recall 0.85–0.86), co wskazuje, że przy umiarkowanych zmianach oświetlenia metody te są w stanie zachować dobrą jakość dopasowań, jednak ustępują metodom bardziej zaawansowanym.

Kolejną analizę wykonano dla sceny zamku, gdzie na klatce 6 względem klatki 1 całkowicie zmieniło się położenie słońca na niebie, obecnie oświetla teren z za zamku. Front zamku jest zacieniony.

In [ ]:
img1_path = katalog_hpatches / "i_fenis" / "1.ppm"
img2_path = katalog_hpatches / "i_fenis" / "6.ppm"

H = np.loadtxt(img1_path.parent / 'H_1_6')

results4 = compare(
    img1_path,
    img2_path,
    H,
    matching,
    device,
    visualize=False
)

SIFT: 72 matches
SIFT+FLANN: 74 matches
ORB: 138 matches
AKAZE: 89 matches
SuperGlue: 281 matches
SIFT       -> Recall: 0.65, MMA: 0.65
SIFT+FLANN -> Recall: 0.65, MMA: 0.65
ORB        -> Recall: 0.00, MMA: 0.00
AKAZE      -> Recall: 0.18, MMA: 0.18
SuperGlue  -> Recall: 0.87, MMA: 0.87


Zwiększenie odstępu między klatkami oraz istotna zmiana oświetlenia (przejście z oświetlonego frontu na zacienienie) powodują wyraźny spadek jakości dopasowań dla większości metod. Największa degradacja widoczna jest dla ORB i AKAZE. ORB osiąga Recall równy 0, co wskazuje na całkowitą utratę zdolności poprawnego dopasowania, natomiast AKAZE uzyskuje bardzo niską skuteczność (Recall 0.18).

SIFT i SIFT+FLANN również ulegają pogorszeniu (Recall 0.65), co pokazuje, że nawet metody oparte na gradientach tracą odporność przy tak dużej zmianie warunków oświetleniowych i układu sceny. Jednak wciąż działają lepiej niż metody binarne.

SuperGlue osiąga najwyższą skuteczność (Recall 0.87) i jako jedyna metoda zachowuje wysoką jakość dopasowań mimo ekstremalnej zmiany oświetlenia. Wynika to z wykorzystania modelu uczonego, który uwzględnia globalne relacje między punktami i jest bardziej odporny na zmiany kontekstu sceny.

#### Pszczoły

W kolejnej analizie porównano dwie klatki sceny (w kolorze) pszczelarza wyciągającego plaster miodu z ula. Na klatce 2 widać względem klatki 1 zmianę perspketywy - pszczelarz jest widoczny bardziej od boku.

In [ ]:
img1_path = katalog_hpatches / "v_bees" / "1.ppm"
img2_path = katalog_hpatches / "v_bees" / "2.ppm"

H = np.loadtxt(img1_path.parent / 'H_1_2')

results3 = compare(
    img1_path,
    img2_path,
    H,
    matching,
    device,
    visualize=False
)

SIFT: 2230 matches
SIFT+FLANN: 2242 matches
ORB: 280 matches
AKAZE: 1912 matches
SuperGlue: 1048 matches
SIFT       -> Recall: 0.96, MMA: 0.96
SIFT+FLANN -> Recall: 0.96, MMA: 0.96
ORB        -> Recall: 0.91, MMA: 0.91
AKAZE      -> Recall: 0.91, MMA: 0.91
SuperGlue  -> Recall: 0.00, MMA: 0.00


Zmiana perspektywy (widok bardziej z boku) nie wpływa istotnie na skuteczność metod SIFT i SIFT+FLANN, które osiągają wysokie wartości Recall (0.96) przy dużej liczbie dopasowań. Podobnie ORB i AKAZE uzyskują dobre wyniki (Recall ≈ 0.91), co wskazuje, że w tej scenie zmiana geometrii nie jest na tyle duża, aby istotnie pogorszyć jakość dopasowania.

AKAZE generuje dużą liczbę dopasowań (1912), przy zachowaniu wysokiej jakości, co sugeruje, że struktura sceny (wiele powtarzalnych, teksturalnych elementów) sprzyja tej metodzie.

SuperGlue, mimo relatywnie dużej liczby dopasowań (1048), uzyskuje Recall równy 0.

W kolejnej analizie porównano dwie klatki sceny (w kolorze) pszczelarza wyciągającego plaster miodu z ula. Na klatce 4 widać względem klatki 1 zmianę perspketywy - pszczelarz jest widoczny bardziej od dołu.

In [ ]:
img1_path = katalog_hpatches / "v_bees" / "1.ppm"
img2_path = katalog_hpatches / "v_bees" / "4.ppm"

H = np.loadtxt(img1_path.parent / 'H_1_4')

results3 = compare(
    img1_path,
    img2_path,
    H,
    matching,
    device,
    visualize=False
)

SIFT: 2079 matches
SIFT+FLANN: 2087 matches
ORB: 280 matches
AKAZE: 1691 matches
SuperGlue: 916 matches
SIFT       -> Recall: 0.97, MMA: 0.97
SIFT+FLANN -> Recall: 0.97, MMA: 0.97
ORB        -> Recall: 0.94, MMA: 0.94
AKAZE      -> Recall: 0.93, MMA: 0.93
SuperGlue  -> Recall: 0.01, MMA: 0.01


Zmiana perspektywy (widok od dołu) nie powoduje istotnej degradacji jakości dopasowań dla metod klasycznych. SIFT i SIFT+FLANN osiągają bardzo wysokie wartości Recall (0.97), co potwierdza ich dużą odporność na zmiany geometrii sceny. ORB i AKAZE również uzyskują wysoką skuteczność (Recall ≈ 0.93–0.94), co wskazuje, że w analizowanym przypadku zmiana perspektywy nie jest na tyle istotna, aby znacząco pogorszyć działanie metod opartych na deskryptorach binarnych.

AKAZE generuje dużą liczbę dopasowań przy zachowaniu wysokiej jakości, co sugeruje, że scena zawiera bogatą strukturę teksturalną, sprzyjającą stabilnej detekcji punktów.

W przypadku SuperGlue, mimo dużej liczby dopasowań (916), uzyskano bardzo niski Recall (0.01).

W kolejnej analizie porównano dwie klatki sceny (w kolorze) pszczelarza wyciągającego plaster miodu z ula. Na klatce 6 widać względem klatki 1 wyraźną zmianę perspketywy - pszczelarz jest widoczny bardzo od boku.

In [ ]:
img1_path = katalog_hpatches / "v_bees" / "1.ppm"
img2_path = katalog_hpatches / "v_bees" / "6.ppm"

H = np.loadtxt(img1_path.parent / 'H_1_6')

results3 = compare(
    img1_path,
    img2_path,
    H,
    matching,
    device,
    visualize=False
)

SIFT: 1126 matches
SIFT+FLANN: 1174 matches
ORB: 202 matches
AKAZE: 1546 matches
SuperGlue: 829 matches
SIFT       -> Recall: 0.85, MMA: 0.85
SIFT+FLANN -> Recall: 0.83, MMA: 0.83
ORB        -> Recall: 0.75, MMA: 0.75
AKAZE      -> Recall: 0.66, MMA: 0.66
SuperGlue  -> Recall: 0.00, MMA: 0.00


Zmiana perspektywy (silny widok z boku) powoduje zauważalny spadek jakości dopasowań dla wszystkich metod klasycznych. SIFT i SIFT+FLANN utrzymują stosunkowo wysoką skuteczność (Recall 0.83–0.85), co potwierdza ich dobrą odporność na zmiany geometrii sceny.

ORB osiąga umiarkowane wyniki (Recall 0.75), natomiast AKAZE uzyskuje niższą skuteczność (Recall 0.66), mimo generowania dużej liczby dopasowań, co wskazuje na większy udział błędnych dopasowań.

Wraz ze wzrostem różnicy perspektywy widoczny jest spadek jakości dopasowań w porównaniu do wcześniejszych przypadków (mniejszych zmian), co potwierdza ograniczoną odporność metod na silne transformacje geometryczne.
W przypadku SuperGlue, pomimo relatywnie dużej liczby dopasowań (829), uzyskano Recall równy 0.

#### Wnioski

Metody oparte na gradientach (SIFT, SIFT+FLANN) wykazują najwyższą stabilność i skuteczność w szerokim zakresie warunków, zarówno przy zmianach perspektywy, jak i oświetlenia. Dzięki wykorzystaniu informacji o gradientach są one odporne na zmiany intensywności jasności oraz umiarkowane transformacje geometryczne, co przekłada się na wysokie wartości Recall w większości testów.

Metody wykorzystujące deskryptory binarne (ORB, AKAZE) charakteryzują się dobrą skutecznością głównie w przypadku zmian perspektywy, jednak ich wydajność znacząco spada przy zmianach oświetlenia. Wynika to z mniejszej odporności deskryptorów binarnych na zmiany intensywności obrazu. Jednocześnie metody te oferują korzystny kompromis między szybkością a jakością dopasowań.

Metoda oparta na modelu uczonym (SuperGlue) wykazuje wysoką odporność na zmiany oświetlenia, co wynika z wykorzystania globalnego kontekstu i uczenia głębokiego. Metoda ta nie radzi sobie z dużymi zmianami perspektywy / orientacji sceny.

### Wpływ wybranych parametrów na skuteczność działania metod

W tej części zbadano wpływ wybranych parametrów dla każdej metody. Testy przeprowadzono na dwóch parach obrazów. Dobrano je tak, aby dla wartości domyślnych parametrów dana metoda dawała średnie wartości Recall i MMA - aby było jasno widać pogorszenie / poprawę wartości. Dodatkowo nie testowano parametrów danej metody na wielu obrazach, ponieważ w tej części celem było sprawdzenie wpływu parametrów, a nie sceny.

In [ ]:
def param_sift(img1_path, img2_path, H, error_thresh=3, visualize=True):

    img1_gray = cv2.imread(str(img1_path), cv2.IMREAD_GRAYSCALE);
    img2_gray = cv2.imread(str(img2_path), cv2.IMREAD_GRAYSCALE);

    ratios = [0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6]

    for ratio in ratios:
      kp1_sift, kp2_sift, matches_sift = run_sift(img1_gray, img2_gray, ratio=ratio, visualize=visualize)
      r_sift, mma_sift = evaluate_matches(kp1_sift, kp2_sift, matches_sift, H, threshold=error_thresh)

      kp1_flann, kp2_flann, matches_flann = run_sift_flann(img1_gray, img2_gray, ratio=ratio, visualize=visualize)
      r_flann, mma_flann = evaluate_matches(kp1_flann, kp2_flann, matches_flann, H, threshold=error_thresh)

      print(f"SIFT       -> Ratio: {ratio:.2f}, Recall: {r_sift:.2f}, MMA: {mma_sift:.2f}")
      print(f"SIFT+FLANN -> Ratio: {ratio:.2f}, Recall: {r_flann:.2f}, MMA: {mma_flann:.2f}")

In [ ]:
def param_superglue(img1_path, img2_path, H, error_thresh=3, visualize=True):

    keypoint_thresholds = [0.001, 0.005, 0.01]
    match_thresholds = [0.1, 0.2, 0.3]

    for kp_th in keypoint_thresholds:
        for match_th in match_thresholds:

            matching = Matching({
                'superpoint': {"nms_radius": 4,"keypoint_threshold": kp_th},
                'superglue': {"weights": 'outdoor',"match_threshold": match_th}
            }).eval().to(device)

            kp1_sg, kp2_sg, matches_sg = run_superglue(img1_path, img2_path, matching, device, visualize=visualize)
            r_sg, mma_sg = evaluate_matches(kp1_sg, kp2_sg, matches_sg, H, threshold=error_thresh)

            print(f"keypoint_threshold={kp_th:.3f}, match_threshold={match_th:.2f} -> Recall: {r_sg:.2f}, MMA: {mma_sg:.2f}")


In [ ]:
img1_path = katalog_hpatches / "i_fenis" / "1.ppm"
img2_path = katalog_hpatches / "i_fenis" / "6.ppm"

H = np.loadtxt(img1_path.parent / 'H_1_6')

param_sift(img1_path, img2_path, H, visualize=False)
param_superglue(img1_path, img2_path, H, visualize=False)


SIFT: 274 matches
SIFT+FLANN: 300 matches
SIFT       -> Ratio: 0.90, Recall: 0.21, MMA: 0.21
SIFT+FLANN -> Ratio: 0.90, Recall: 0.19, MMA: 0.19
SIFT: 135 matches
SIFT+FLANN: 146 matches
SIFT       -> Ratio: 0.85, Recall: 0.39, MMA: 0.39
SIFT+FLANN -> Ratio: 0.85, Recall: 0.36, MMA: 0.36
SIFT: 97 matches
SIFT+FLANN: 103 matches
SIFT       -> Ratio: 0.80, Recall: 0.53, MMA: 0.53
SIFT+FLANN -> Ratio: 0.80, Recall: 0.50, MMA: 0.50
SIFT: 72 matches
SIFT+FLANN: 72 matches
SIFT       -> Ratio: 0.75, Recall: 0.65, MMA: 0.65
SIFT+FLANN -> Ratio: 0.75, Recall: 0.65, MMA: 0.65
SIFT: 62 matches
SIFT+FLANN: 63 matches
SIFT       -> Ratio: 0.70, Recall: 0.71, MMA: 0.71
SIFT+FLANN -> Ratio: 0.70, Recall: 0.71, MMA: 0.71
SIFT: 48 matches
SIFT+FLANN: 49 matches
SIFT       -> Ratio: 0.65, Recall: 0.85, MMA: 0.85
SIFT+FLANN -> Ratio: 0.65, Recall: 0.84, MMA: 0.84
SIFT: 41 matches
SIFT+FLANN: 43 matches
SIFT       -> Ratio: 0.60, Recall: 0.90, MMA: 0.90
SIFT+FLANN -> Ratio: 0.60, Recall: 0.88, MMA: 0.88
L

W przypadku metody SIFT i SIFT+FLANN wraz ze zmniejszaniem parametru ratio obserwowany jest spadek liczby dopasowań przy jednoczesnym wzroście wartości Recall i MMA. Oznacza to, że bardziej restrykcyjny test Lowe’a eliminuje błędne dopasowania i poprawia ogólną jakość wyników. Najlepsze rezultaty jakościowe uzyskano dla niższych wartości ratio (0.6). Różnice między SIFT a SIFT+FLANN są niewielkie.

W przypadku metody SuperGlue zwiększanie parametru match_threshold prowadzi do spadku liczby dopasowań przy jednoczesnym wzroście Recall i MMA. Oznacza to skuteczniejsze filtrowanie słabych dopasowań i poprawę ich jakości. Parametr keypoint_threshold wpływa głównie na liczbę wykrywanych punktów i w mniejszym stopniu na jakość dopasowań, co widać w stosunkowo stabilnych wartościach Recall w badanym zakresie.

SIFT wykazuje dużą wrażliwość na parametr ratio, który wpływa na kompromis między liczbą a jakością dopasowań. SuperGlue jest bardziej stabilny jakościowo i mniej wrażliwy na zmiany parametrów, ponieważ korzysta z modelu uczonego, który uwzględnia kontekst globalny podczas dopasowania punktów.

In [ ]:
def param_orb(img1_path, img2_path, H, error_thresh=3, visualize=False):

    img1_gray = cv2.imread(str(img1_path), cv2.IMREAD_GRAYSCALE)
    img2_gray = cv2.imread(str(img2_path), cv2.IMREAD_GRAYSCALE)

    thresholds = [10, 20, 40]
    nfeatures_list = [200, 500, 1000]

    for nfeatures in nfeatures_list:
        for fasttreshold in thresholds:

            kp1_orb, kp2_orb, matches_orb = run_orb(img1_gray, img2_gray,nfeatures=nfeatures, fasttreshold=fasttreshold, visualize=visualize)
            r_orb, mma_orb = evaluate_matches(kp1_orb, kp2_orb, matches_orb, H, threshold=error_thresh)

            print(f"ORB -> nfeatures={nfeatures}, fastThreshold={fasttreshold} -> Recall: {r_orb:.2f}, MMA: {mma_orb:.2f}")

In [ ]:
def param_akaze(img1_path, img2_path, H, error_thresh=3, visualize=False):

    img1_gray = cv2.imread(str(img1_path), cv2.IMREAD_GRAYSCALE)
    img2_gray = cv2.imread(str(img2_path), cv2.IMREAD_GRAYSCALE)

    thresholds = [0.0005, 0.001, 0.005]
    noctaves_list = [2, 4, 6]

    for noctaves in noctaves_list:
        for treshold in thresholds:

            kp1_akaze, kp2_akaze, matches_akaze = run_akaze(img1_gray, img2_gray,noctaves=noctaves, treshold=treshold,visualize=visualize)
            r_akaze, mma_akaze = evaluate_matches(kp1_akaze, kp2_akaze, matches_akaze,H, threshold=error_thresh)

            print(f"AKAZE -> nOctaves={noctaves}, threshold={treshold} -> Recall: {r_akaze:.2f}, MMA: {mma_akaze:.2f}")

In [ ]:
img1_path = katalog_hpatches / "v_boat" / "1.ppm"
img2_path = katalog_hpatches / "v_boat" / "5.ppm"

H = np.loadtxt(img1_path.parent / 'H_1_5')

param_orb(img1_path, img2_path, H, visualize=False)
param_akaze(img1_path, img2_path, H, visualize=False)

ORB: 82 matches
ORB -> nfeatures=200, fastThreshold=10 -> Recall: 0.62, MMA: 0.62
ORB: 81 matches
ORB -> nfeatures=200, fastThreshold=20 -> Recall: 0.63, MMA: 0.63
ORB: 81 matches
ORB -> nfeatures=200, fastThreshold=40 -> Recall: 0.63, MMA: 0.63
ORB: 186 matches
ORB -> nfeatures=500, fastThreshold=10 -> Recall: 0.66, MMA: 0.66
ORB: 185 matches
ORB -> nfeatures=500, fastThreshold=20 -> Recall: 0.66, MMA: 0.66
ORB: 185 matches
ORB -> nfeatures=500, fastThreshold=40 -> Recall: 0.66, MMA: 0.66
ORB: 372 matches
ORB -> nfeatures=1000, fastThreshold=10 -> Recall: 0.63, MMA: 0.63
ORB: 373 matches
ORB -> nfeatures=1000, fastThreshold=20 -> Recall: 0.63, MMA: 0.63
ORB: 372 matches
ORB -> nfeatures=1000, fastThreshold=40 -> Recall: 0.63, MMA: 0.63
AKAZE: 1583 matches
AKAZE -> nOctaves=2, threshold=0.0005 -> Recall: 0.46, MMA: 0.46
AKAZE: 1178 matches
AKAZE -> nOctaves=2, threshold=0.001 -> Recall: 0.51, MMA: 0.51
AKAZE: 332 matches
AKAZE -> nOctaves=2, threshold=0.005 -> Recall: 0.59, MMA: 0.59
A

W przypadku ORB zmiana parametru fastThreshold nie powoduje istotnych zmian w wartościach Recall i MMA. Natomiast zwiększanie parametru nfeatures prowadzi do wzrostu liczby dopasowań, jednak najlepsze wartości uzyskano dla wartości 500, co oznacza, że ani nadmiar, ani niedomiar nie są najlepszym wyborem.

W przypadku AKAZE parametr threshold ma istotny wpływ na wyniki. Zmniejszenie jego wartości zwiększa liczbę wykrywanych punktów charakterystycznych, co przekłada się na większą liczbę dopasowań i potencjalnie wyższy Recall, jednak kosztem większej liczby błędów. Z kolei zwiększenie wartości threshold powoduje bardziej restrykcyjną detekcję punktów, co może poprawić jakość dopasowań.

Zbyt mała liczba oktaw prowadzi do pogorszenia wartości Recall i MMA.

ORB jest mniej wrażliwy na zmianę parametrów i daje bardziej stabilne, ale przeciętne wyniki, natomiast AKAZE wykazuje większą zależność od parametrów i pozwala lepiej kontrolować kompromis między liczbą a jakością dopasowań.

#### Podsumowanie

Dobór parametrów ma istotny wpływ na jakość dopasowań, jednak jego znaczenie zależy od zastosowanej metody. Metody klasyczne, szczególnie SIFT i AKAZE, pozwalają na precyzyjną kontrolę kompromisu między liczbą a jakością dopasowań. ORB pozostaje metodą mniej wrażliwą na parametry, oferując stabilne, lecz przeciętne wyniki. SuperGlue, dzięki wykorzystaniu uczenia głębokiego, wykazuje największą stabilność i najmniejszą zależność od parametrów.